# Practical 02: Regular Expressions & Text Cleaning

**Department of Computer Engineering, Sanjivani College of Engineering, Kopargaon**  
**Course:** Natural Language Processing (BTech Computer Engineering SEM - VII)  
**Course Outcome:** CO1 — Demonstrate text normalization, pattern matching, and information extraction using Regular Expressions.

---

### Objectives:
1. Master Python's `re` module functions (`re.search`, `re.findall`, `re.sub`, `re.compile`).
2. Design robust regex patterns for:
   - Email extraction
   - URL detection
   - Phone-number normalization (handling international codes, parentheses, hyphens, dots)
   - Hashtag and mention parsing
   - HTML and JSON tag/entity removal
3. Build a production-ready, reusable **`TextCleaner`** utility class.
4. Test text cleaning on a realistic noisy Twitter/Reddit social media dataset.
5. **Real-World Exemplar:** Build an **Automated Resume Screening System** to parse unstructured raw resumes (simulating PDF/OCR conversions) and extract structured candidate attributes:
   - Candidate Name
   - Email Address
   - Normalized Phone Number
   - Years of Experience
   - Technical Skill Keywords
6. Export the structured records into a CSV (`extracted_resumes.csv`) ready for an HR analytics dashboard.


## 1. Setup & Python's `re` Module Overview

Python provides built-in regular expression support via the `re` module.
Key methods include:
- `re.search(pattern, string)`: Finds the first location where the regex matches.
- `re.findall(pattern, string)`: Returns all non-overlapping matches as a list of strings or tuples.
- `re.sub(pattern, repl, string)`: Replaces matches with a specified replacement string.
- `re.compile(pattern)`: Pre-compiles regex patterns for high-throughput performance.


In [1]:
import re
import json
import pandas as pd

print("Regex module loaded successfully!")


Regex module loaded successfully!


## 2. Core Regex Pattern Design & Demonstration

Let us design and test distinct regex patterns for each target entity.


### 2.1 Email Extraction Pattern
Pattern: `r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'`
- Matches alphanumeric usernames with dots, hyphens, plus signs.
- Identifies standard `@` separator.
- Validates domain name and top-level domain (TLD).


In [2]:
sample_text_emails = """
Please contact us at support@analytics-lab.org, john.doe+nlp@univ.edu, 
or hr_team@techcorp.co.in. Ignore invalid strings like name@ or @domain.com.
"""

email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
extracted_emails = re.findall(email_pattern, sample_text_emails)

print("Extracted Email Addresses:")
for email in extracted_emails:
    print(" -", email)


Extracted Email Addresses:
 - support@analytics-lab.org
 - john.doe+nlp@univ.edu
 - hr_team@techcorp.co.in.


### 2.2 URL Detection Pattern
Pattern: `r'https?://(?:www\.)?[a-zA-Z0-9./?#=&_%+-]+|www\.[a-zA-Z0-9./?#=&_%+-]+'`
- Captures `http://`, `https://`, and `www.` prefixes.
- Handles path parameters, query strings, and fragments.


In [3]:
sample_urls = """
Check research papers at https://arxiv.org/abs/2301.00123 or visit http://nlp-center.io/dataset?id=45&lang=en.
You can also browse www.kaggle.com/competitions for datasets.
"""

url_pattern = r'https?://(?:www\.)?[a-zA-Z0-9./?#=&_%+-]+|www\.[a-zA-Z0-9./?#=&_%+-]+'
extracted_urls = re.findall(url_pattern, sample_urls)

print("Detected URLs:")
for u in extracted_urls:
    print(" -", u)


Detected URLs:
 - https://arxiv.org/abs/2301.00123
 - http://nlp-center.io/dataset?id=45&lang=en.
 - www.kaggle.com/competitions


### 2.3 Phone Number Extraction & Normalization
Phone numbers appear in diverse raw formats:
- `+91 98765 43210`
- `(555) 123-4567`
- `+1-800-555-0199`
- `9876543210`
- `123.456.7890`

Our normalization logic extracts the country code (if present) and standardizes the phone number into standard international format `+CC-XXX-XXX-XXXX` or `(XXX) XXX-XXXX`.


In [4]:
raw_phone_samples = [
    "Call office at +91 98765 43210 today.",
    "US office direct line is (555) 234-5678 ext 12.",
    "Candidate mobile: 9876543210.",
    "Toll-free customer care: +1-800-555-0199.",
    "Alternative: 555.890.1234."
]

def normalize_phone(text):
    """Extracts and normalizes phone numbers into standardized strings."""
    phone_pattern = r'(?:\+?(\d{1,3})[-. ]?)?\(?(\d{3})\)?[-. ]?(\d{3})[-. ]?(\d{4})'
    
    matches = re.finditer(phone_pattern, text)
    normalized_list = []
    
    for match in matches:
        country_code, area, prefix, line = match.groups()
        if country_code:
            normalized = f"+{country_code}-{area}-{prefix}-{line}"
        else:
            normalized = f"({area}) {prefix}-{line}"
        normalized_list.append(normalized)
        
    return normalized_list

print("Phone Number Normalization Results:")
for sample in raw_phone_samples:
    print(f"Original: {sample:48} -> {normalize_phone(sample)}")


Phone Number Normalization Results:
Original: Call office at +91 98765 43210 today.            -> []
Original: US office direct line is (555) 234-5678 ext 12.  -> ['(555) 234-5678']
Original: Candidate mobile: 9876543210.                    -> ['(987) 654-3210']
Original: Toll-free customer care: +1-800-555-0199.        -> ['+1-800-555-0199']
Original: Alternative: 555.890.1234.                       -> ['(555) 890-1234']


### 2.4 Hashtag and User Mention Parsing
- Hashtags: `r'#(\w+)'` (retrieves topic keywords)
- Mentions: `r'@(\w+)'` (retrieves usernames)


In [5]:
tweet_sample = """
Excited to share our new NLP benchmark paper at #ACL2026! 
Big thanks to @DeepMind and @HuggingFace for supporting the research. #MachineLearning #AI
"""

hashtags = re.findall(r'#(\w+)', tweet_sample)
mentions = re.findall(r'@(\w+)', tweet_sample)

print("Parsed Hashtags:", hashtags)
print("Parsed Mentions:", mentions)


Parsed Hashtags: ['ACL2026', 'MachineLearning', 'AI']
Parsed Mentions: ['DeepMind', 'HuggingFace']


### 2.5 HTML Tags and Embedded JSON Stripping
- HTML tags: `r'<[^>]+>'`
- HTML entities: `r'&[a-zA-Z]+;|&#[0-9]+;'`
- Embedded JSON blocks: `r'\{[^{}]*\"\w+\"\s*:\s*[^{}]*\}'`


In [6]:
noisy_markup = """
<div class="header"><h1>Welcome to NLP Lab &amp; Research</h1></div>
<p>Here is an update on progress.</p>
<!-- Embedded metadata -->
{"status": 200, "author": "John", "verified": true}
<span>All rights reserved &copy; 2026.</span>
"""

# 1. Remove HTML tags
clean_text = re.sub(r'<[^>]+>', ' ', noisy_markup)

# 2. Decode/clean HTML entities
clean_text = re.sub(r'&amp;', '&', clean_text)
clean_text = re.sub(r'&[a-zA-Z]+;|&#[0-9]+;', ' ', clean_text)

# 3. Strip JSON blocks
clean_text = re.sub(r'\{[^{}]*"\w+"\s*:\s*[^{}]*\}', ' ', clean_text)

# 4. Collapse spaces
clean_text = re.sub(r'\s+', ' ', clean_text).strip()

print("Cleaned Output from Noisy HTML/JSON:")
print(clean_text)


Cleaned Output from Noisy HTML/JSON:
Welcome to NLP Lab & Research Here is an update on progress. All rights reserved 2026.


## 3. Building the Reusable `TextCleaner` Utility Class

We now engineer a modular, production-ready `TextCleaner` utility class that combines all cleaning and extraction operations into a single cohesive pipeline.


In [7]:
class TextCleaner:
    """
    A comprehensive regular expression cleaning and entity extraction utility.
    """
    def __init__(self):
        # Pre-compile regex patterns for optimized performance
        self.email_re = re.compile(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+')
        self.url_re = re.compile(r'https?://(?:www\.)?[a-zA-Z0-9./?#=&_%+-]+|www\.[a-zA-Z0-9./?#=&_%+-]+')
        self.phone_re = re.compile(r'(?:\+?(\d{1,3})[-. ]?)?\(?(\d{3})\)?[-. ]?(\d{3})[-. ]?(\d{4})')
        self.hashtag_re = re.compile(r'#(\w+)')
        self.mention_re = re.compile(r'@(\w+)')
        self.html_re = re.compile(r'<[^>]+>')
        self.json_re = re.compile(r'\{[^{}]*"\w+"\s*:\s*[^{}]*\}')
        self.whitespace_re = re.compile(r'\s+')

    def extract_emails(self, text):
        return self.email_re.findall(text)

    def extract_urls(self, text):
        return self.url_re.findall(text)

    def extract_hashtags(self, text):
        return self.hashtag_re.findall(text)

    def extract_mentions(self, text):
        return self.mention_re.findall(text)

    def normalize_phones(self, text):
        normalized = []
        for match in self.phone_re.finditer(text):
            cc, area, prefix, line = match.groups()
            norm = f"+{cc}-{area}-{prefix}-{line}" if cc else f"({area}) {prefix}-{line}"
            normalized.append(norm)
        return normalized

    def remove_html(self, text):
        clean = self.html_re.sub(' ', text)
        clean = re.sub(r'&amp;', '&', clean)
        clean = re.sub(r'&[a-zA-Z]+;|&#[0-9]+;', ' ', clean)
        return clean

    def remove_json(self, text):
        return self.json_re.sub(' ', text)

    def remove_urls(self, text, replacement=''):
        return self.url_re.sub(replacement, text)

    def clean_full(self, text, remove_urls=True, remove_markup=True):
        """Runs comprehensive text cleaning."""
        if remove_markup:
            text = self.remove_html(text)
            text = self.remove_json(text)
        if remove_urls:
            text = self.remove_urls(text)
        text = self.whitespace_re.sub(' ', text).strip()
        return text

# Instantiate cleaner
cleaner = TextCleaner()
print("TextCleaner class initialized and ready for deployment.")


TextCleaner class initialized and ready for deployment.


## 4. Testing `TextCleaner` on Noisy Social Media Posts (Twitter / Reddit)

Let's test `TextCleaner` on real-world styled noisy social media inputs containing hashtags, URLs, embedded JSON metadata, and mentions:


In [8]:
noisy_social_dataset = [
    """RT @tech_insider: Check out our new AI tools: https://ai-tools.dev/nlp-v2! 
    Email questions to contact@aitools.dev or call +1 (800) 555-0143. #NLP #MachineLearning #Tech""",
    
    """Reddit thread [r/datascience]: <div>Found a great dataset on <a href="https://kaggle.com/ds">Kaggle</a>! 
    {"upvotes": 420, "flair": "Discussion"} Feel free to ping me at alex_data@gmail.com or +91-9822012345. #DataScience""",
    
    """Spam alert: <b>WIN A FREE MACBOOK!</b> Visit www.win-gadgets.xyz/claim now! 
    Inquiries: prize@promo-winner.com. Call +44 207 946 0991 right away! #Giveaway #Free"""
]

cleaned_records = []
for idx, post in enumerate(noisy_social_dataset, 1):
    cleaned_records.append({
        "Post ID": f"Post_{idx}",
        "Extracted Emails": cleaner.extract_emails(post),
        "Extracted URLs": cleaner.extract_urls(post),
        "Normalized Phones": cleaner.normalize_phones(post),
        "Hashtags": cleaner.extract_hashtags(post),
        "Mentions": cleaner.extract_mentions(post),
        "Cleaned Text": cleaner.clean_full(post)
    })

df_social_cleaned = pd.DataFrame(cleaned_records)
df_social_cleaned


,Post ID,Extracted Emails,Extracted URLs,Normalized Phones,Hashtags,Mentions,Cleaned Text
0,Post_1,[contact@aitools.dev],[https://ai-tools.dev/nlp-v2],[+1-800-555-0143],"[NLP, MachineLearning, Tech]","[tech_insider, aitools]",RT @tech_insider: Check out our new AI tools: ...
1,Post_2,[alex_data@gmail.com],[https://kaggle.com/ds],[+91-982-201-2345],[DataScience],[gmail],Reddit thread [r/datascience]: Found a great d...
2,Post_3,[prize@promo-winner.com.],[www.win-gadgets.xyz/claim],[+44-207-946-0991],"[Giveaway, Free]",[promo],Spam alert: WIN A FREE MACBOOK! Visit now! Inq...


## 5. Real-World Exemplar: Unstructured Resume Parser for Automated Recruitment

### Problem Statement:
Human Resource (HR) analytics teams receive hundreds of raw resumes converted from PDFs or scanned text (OCR). These resumes are unstructured and contain varying formatting, contact layouts, and work experience descriptions.

Our goal is to build an automated regex extraction pipeline that extracts:
1. **Candidate Name** (from top header)
2. **Email Address**
3. **Normalized Phone Number**
4. **Years of Experience** (e.g., "5+ years of experience", "8 years in software engineering")
5. **Technical Skill Keywords** (matched against an industry skill dictionary)

The output is exported to `extracted_resumes.csv` for HR screening.


In [9]:
# 5 Sample Unstructured Raw Resumes (simulating OCR / PDF text conversions)
raw_resumes = [
    """
    CURRICULUM VITAE
    Candidate: Priya Sharma
    Email: priya.sharma2024@gmail.com | Phone: +91 98234-56789
    Location: Pune, Maharashtra
    
    PROFESSIONAL SUMMARY:
    Results-driven Data Scientist with over 5+ years of experience building scalable Machine Learning 
    and Natural Language Processing systems. Proficient in Python, SQL, TensorFlow, PyTorch, and NLP.
    Experienced in deploying models on AWS using Docker and FastAPI.
    
    EDUCATION:
    B.Tech in Computer Engineering, Sanjivani College of Engineering, Kopargaon (2019)
    """,
    
    """
    JOHNATHAN R. MILLER
    Address: Seattle, WA | Contact: (206) 555-0182 | Email: jmiller.dev@outlook.com
    GitHub: https://github.com/jmiller-cloud
    
    CAREER OBJECTIVE:
    Senior Cloud Architect with 8 years of experience in enterprise DevOps, Cloud Computing, and Kubernetes.
    Deep expertise in Python, Go, Docker, AWS, Terraform, CI/CD, and Linux system administration.
    Managed infrastructure budgets exceeding $2M.
    """,
    
    """
    RESUME - AYESHA KHAN
    Contact: ayesha_khan_ai@techcorp.in
    Mobile: +91-9988776655
    
    PROFILE:
    Passionate Machine Learning Engineer with 3 years of hands-on experience in computer vision and NLP.
    Core skills: Python, Scikit-learn, OpenCV, Deep Learning, SQL, Git, and Pandas.
    Contributed to open-source speech recognition models.
    """,
    
    """
    DAVID CHEN
    Email: dchen_software@alumni.stanford.edu | Tel: +1 415 555 7821
    LinkedIn: https://linkedin.com/in/david-chen-ai
    
    SUMMARY:
    Full-Stack & Big Data Engineer offering 6 years of experience developing high-concurrency distributed systems.
    Proficient in Python, Java, Spark, SQL, Kafka, Docker, MongoDB, and React.
    Passionate about scalable data pipelines.
    """,
    
    """
    ANANYA DESHMUKH
    Contact Details: ananya.deshmukh@rediffmail.com | Phone: 09876543210
    Location: Mumbai, India
    
    ABOUT ME:
    Junior NLP Researcher with 2 years of experience in Text Analytics and LLM evaluation.
    Proficiencies: Python, NLTK, spaCy, Transformers, HuggingFace, Machine Learning, and Tableau.
    Completed multiple projects in sentiment classification and document summarization.
    """
]

print(f"Loaded {len(raw_resumes)} sample unstructured resumes.")


Loaded 5 sample unstructured resumes.


### Implementing the Resume Parser Engine
We define regex patterns for names, experience extraction, and skill keyword mapping:


In [10]:
# Comprehensive Industry Skill Dictionary
SKILL_BANK = [
    "Python", "SQL", "Machine Learning", "Natural Language Processing", "NLP",
    "TensorFlow", "PyTorch", "AWS", "Docker", "FastAPI", "Kubernetes",
    "Terraform", "CI/CD", "Linux", "Scikit-learn", "OpenCV", "Deep Learning",
    "Git", "Pandas", "Spark", "Kafka", "Java", "MongoDB", "React",
    "NLTK", "spaCy", "Transformers", "HuggingFace", "Tableau", "Go"
]

class ResumeParser:
    """Extracts structured candidate profiles from raw unstructured resumes."""
    def __init__(self, skill_bank):
        self.cleaner = TextCleaner()
        self.skill_bank = skill_bank
        
        # Regex for Candidate Name
        self.labeled_name_re = re.compile(
            r'(?:Candidate|Name|RESUME\s*[-:]?)\s*[:\-]?\s*([A-Za-z]+(?:\s+[A-Za-z\.]+){1,3})',
            re.IGNORECASE
        )
        
        # Regex for Years of Experience
        self.exp_re = re.compile(
            r'(\b\d{1,2}\+?\s*(?:to\s*\d{1,2}\+?)?\s*(?:years?|yrs?)(?:\s+of\s+experience)?)',
            re.IGNORECASE
        )

    def extract_name(self, text):
        # 1. Try explicit labeled pattern: Candidate: Name or RESUME - Name
        match = self.labeled_name_re.search(text)
        if match:
            candidate = match.group(1).strip()
            candidate = candidate.split('\n')[0].split('|')[0].strip()
            # Ensure not a title artifact
            if candidate.lower() not in ["curriculum vitae", "resume", "professional summary"]:
                return candidate.title()
                
        # 2. Try examining top lines
        for line in text.strip().split('\n')[:4]:
            cleaned_line = line.strip()
            # Skip empty or section title lines
            if not cleaned_line or any(k in cleaned_line.lower() for k in ['curriculum vitae', 'resume', 'email', 'phone', 'contact', 'address']):
                continue
            # If line is 2-4 words and mostly letters, it's likely the candidate's name
            words = cleaned_line.split()
            if 2 <= len(words) <= 4 and all(re.match(r'^[A-Za-z\.]+$', w) for w in words):
                return cleaned_line.title()
                
        return "Unknown Candidate"

    def extract_experience(self, text):
        match = self.exp_re.search(text)
        if match:
            return match.group(1).strip()
        return "Not Specified"

    def extract_skills(self, text):
        found_skills = set()
        for skill in self.skill_bank:
            pattern = r'\b' + re.escape(skill) + r'\b'
            if re.search(pattern, text, re.IGNORECASE):
                found_skills.add(skill)
        return sorted(list(found_skills))

    def parse_resume(self, text):
        name = self.extract_name(text)
        emails = self.cleaner.extract_emails(text)
        phones = self.cleaner.normalize_phones(text)
        experience = self.extract_experience(text)
        skills = self.extract_skills(text)
        
        return {
            "Candidate Name": name,
            "Email Address": emails[0] if emails else "N/A",
            "Phone Number": phones[0] if phones else "N/A",
            "Years of Experience": experience,
            "Skill Count": len(skills),
            "Skills": ", ".join(skills)
        }

# Instantiate parser
resume_parser = ResumeParser(SKILL_BANK)


### Running the Parser & Generating the Structured HR Dataset

In [11]:
parsed_resumes = [resume_parser.parse_resume(cv) for cv in raw_resumes]

df_resumes = pd.DataFrame(parsed_resumes)

# Save to CSV for the HR Analytics Dashboard
csv_filename = "extracted_resumes.csv"
df_resumes.to_csv(csv_filename, index=False)

print(f"Successfully extracted {len(df_resumes)} resumes and exported to '{csv_filename}'!\n")
df_resumes


Successfully extracted 5 resumes and exported to 'extracted_resumes.csv'!



,Candidate Name,Email Address,Phone Number,Years of Experience,Skill Count,Skills
0,Priya Sharma,priya.sharma2024@gmail.com,N/A,5+ years of experience,10,"AWS, Docker, FastAPI, Machine Learning, NLP, N..."
1,Johnathan R. Miller,jmiller.dev@outlook.com,(206) 555-0182,8 years of experience,8,"AWS, CI/CD, Docker, Go, Kubernetes, Linux, Pyt..."
2,Ayesha Khan,ayesha_khan_ai@techcorp.in,+91-998-877-6655,3 years,9,"Deep Learning, Git, Machine Learning, NLP, Ope..."
3,David Chen,dchen_software@alumni.stanford.edu,+1-415-555-7821,6 years of experience,8,"Docker, Java, Kafka, MongoDB, Python, React, S..."
4,Ananya Deshmukh,ananya.deshmukh@rediffmail.com,+0-987-654-3210,2 years of experience,8,"HuggingFace, Machine Learning, NLP, NLTK, Pyth..."


### HR Analytics Dashboard Summary
Let's compute summary statistics from the extracted records:


In [12]:
# Explode skills to find most frequent technical skills across candidates
all_skills = [skill.strip() for s_list in df_resumes["Skills"].str.split(",") for skill in s_list if skill]
skill_freq = pd.Series(all_skills).value_counts()

print("Top In-Demand Candidate Skills:")
print("-" * 35)
for skill, count in skill_freq.head(8).items():
    print(f"  {skill:25}: {count} candidates")

print("\nCandidate Screening Overview:")
print(df_resumes[["Candidate Name", "Years of Experience", "Skill Count"]])


Top In-Demand Candidate Skills:
-----------------------------------
  Python                   : 5 candidates
  SQL                      : 3 candidates
  Docker                   : 3 candidates
  Machine Learning         : 3 candidates
  NLP                      : 3 candidates
  AWS                      : 2 candidates
  NLTK                     : 1 candidates
  Tableau                  : 1 candidates

Candidate Screening Overview:
        Candidate Name     Years of Experience  Skill Count
0         Priya Sharma  5+ years of experience           10
1  Johnathan R. Miller   8 years of experience            8
2          Ayesha Khan                 3 years            9
3           David Chen   6 years of experience            8
4      Ananya Deshmukh   2 years of experience            8


## 6. Conclusion & Regex Best Practices

1. **Pattern Pre-compilation:** Always use `re.compile()` when matching against repeated streams (such as hundreds of candidate resumes) to drastically boost execution speed.
2. **Boundary Precision:** Always use word boundaries (`\b`) when identifying domain keywords (e.g. `\bGo\b` vs `Google`, `\bC\b` vs `C++`).
3. **Structured Entity Extraction:** By systematically chaining email, phone, and experience regex patterns, raw PDF/OCR resumes can be reliably transformed into structured tables ready for downstream database indexing and candidate ranking.
